# 2. Model and prompting-method selection

Calculate condition means from the blinded human ratings of the 50-item panel.
The supplied annotation workbook has two raters and 96 scores for 48 items.
Two generated items have no recorded scores.

Rank the observed means and show the recorded author choice. Cases differed
across conditions, so the means are descriptive rather than a matched-case
comparison.

In [ ]:
import json
import tempfile
from pathlib import Path

from IPython.display import display
from pandas import DataFrame

from fairforget.dataset_workflow import find_project, load_inputs
from fairforget.io import write_json

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = Path(tempfile.mkdtemp(prefix="warbias-dataset-"))

In [ ]:
from fairforget.dataset_workflow import recorded_selection

analysis, selection_record = recorded_selection(project, inputs)
columns = [
    "rank",
    "candidate_id",
    "strategy",
    "generated_items",
    "rated_items",
    "ratings",
    "mean_score",
]
display(DataFrame(analysis["condition_summary"])[columns])
analysis["coverage"]

The arithmetic mean uses both overall quality scores for each reviewed item.
Lapa has the highest observed mean with both prompting methods: 107 / 24 = 4.4583.
The scores do not distinguish sequential from zero-shot prompting.
The recorded author choice is Lapa with sequential prompting for control over
the three sentence roles. This choice is separate from the tied score ranking.

In [ ]:
write_json(work / "rating-analysis.json", analysis)
write_json(work / "selected-condition.json", selection_record)
result = {
    "stage": "model_selection",
    "selection_record_path": str(work / "selected-condition.json"),
    "selection": analysis["recorded_selection"],
    "highest_mean_conditions": analysis["highest_mean_conditions"],
    "coverage": analysis["coverage"],
}
result